In [1]:
import os
import cv2
import mediapipe as mp

# Paths
image_root = 'dataset/images/train2'
label_root = 'dataset/labels/train2'

# Gesture class names (must match order in your dataset.yaml)
labels = [
    'yes', 'no', 'me', 'you', 'hello', 'hi', 'good',
    'ok', 'sorry', 'i love you', 'thank you', 'welcome'
]
label_map = {name: idx for idx, name in enumerate(labels)}

# Init MediaPipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1)
image_count = 0
labeled_count = 0

for gesture in labels:
    gesture_img_dir = os.path.join(image_root, gesture)
    gesture_label_dir = os.path.join(label_root, gesture)
    os.makedirs(gesture_label_dir, exist_ok=True)

    for img_file in os.listdir(gesture_img_dir):
        if not img_file.lower().endswith(('.jpg', '.png')):
            continue

        img_path = os.path.join(gesture_img_dir, img_file)
        label_path = os.path.join(gesture_label_dir, img_file.replace('.jpg', '.txt').replace('.png', '.txt'))

        image = cv2.imread(img_path)
        if image is None:
            continue

        image_count += 1
        h, w, _ = image.shape
        results = hands.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if not results.multi_hand_landmarks:
            print(f"❌ No hand found in: {img_path}")
            continue

        # Get bounding box from landmarks
        hand_landmarks = results.multi_hand_landmarks[0]
        xs = [lm.x for lm in hand_landmarks.landmark]
        ys = [lm.y for lm in hand_landmarks.landmark]

        xmin = max(min(xs), 0)
        xmax = min(max(xs), 1)
        ymin = max(min(ys), 0)
        ymax = min(max(ys), 1)

        # Convert to YOLO format
        x_center = (xmin + xmax) / 2
        y_center = (ymin + ymax) / 2
        box_width = xmax - xmin
        box_height = ymax - ymin

        # Scale to absolute image dimensions
        yolo_box = [
            label_map[gesture],
            round(x_center, 6),
            round(y_center, 6),
            round(box_width, 6),
            round(box_height, 6)
        ]

        with open(label_path, 'w') as f:
            f.write(" ".join(map(str, yolo_box)) + "\n")

        labeled_count += 1

print(f"\n✅ Processed {image_count} images")
print(f"📝 Created {labeled_count} label files with MediaPipe auto-labeling.")



✅ Processed 1200 images
📝 Created 1200 label files with MediaPipe auto-labeling.
